# Aspect Categorization

This notebook demonstrates how the unique aspects extracted by the **Hybrid ABSA + LLM** pipeline were categorized into higher-level semantic groups.

After processing reviews from four restaurants, the system identified **more than 2000 unique aspects**. While these fine-grained aspects are useful for detailed analysis, they make statistical analysis and comparison between restaurants more difficult.

To simplify the downstream analysis, all unique aspects were grouped into the following **16 categories**:

- `specific_dish`
- `restaurant_general`
- `food_quality`
- `atmosphere`
- `service`
- `drinks`
- `menu_variety`
- `seating_comfort`
- `price_value`
- `staff_behavior`
- `desserts`
- `location_access`
- `waiting_time`
- `reservation`
- `cleanliness`
- `others`

The categorization was performed using an LLM, which assigned each unique aspect to exactly one predefined category.

> **Note**
>
> There is no need to run this notebook again. The categorization has already been completed, and the resulting labels are stored in the Kaggle dataset under the `aspect_category` column.
>
> Running this notebook again requires an OpenAI API key and will generate additional API costs.

In [17]:
pip install -U openai pydantic pandas tqdm

   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 1.7/1.7 MB 29.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 2.1/2.1 MB 38.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
   ----------------------------- ---------- 7.3/9.8 MB 34.9 MB/s eta 0:00:01
   ---------------------------------------- 9.8/9.8 MB 34.1 MB/s eta 0:00:00
  Attempting uninstall: tqdm
    Found existing installation: tqdm 4.66.5
    Uninstalling tqdm-4.66.5:
      Successfully uninstalled tqdm-4.66.5
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.41.5
    Uninstalling pydantic_core-2.41.5:
      Successfully uninstalled pydantic_core-2.41.5
  Attempting uninstall: pydantic
    Found existing installation: pydantic 2.12.5
    Uninstalling pydantic-2.12.5:
      Successfully uninstalled pydantic

  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
aiogram 3.22.0 requires pydantic<2.12,>=2.4.1, but you have pydantic 2.13.4 which is incompatible.
facenet-pytorch 2.6.0 requires numpy<2.0.0,>=1.24.0, but you have numpy 2.4.2 which is incompatible.
pysr 1.5.8 requires pandas<3.0.0,>=0.21.0, but you have pandas 3.0.5 which is incompatible.
streamlit 1.55.0 requires pandas<3,>=1.4.0, but you have pandas 3.0.5 which is incompatible.


In [1]:
pip install kagglehub

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import kagglehub

path = kagglehub.dataset_download(
    "achilov15/french-res-processed/versions/2"
)

print("Path:", path)
df = pd.read_csv(f"{path}/classified_df.csv")

df.head()

Path: C:\Users\timur\.cache\kagglehub\datasets\achilov15\french-res-processed\versions\2


,Unnamed: 0,restaurant,id,text,stars,aspect,polarity,proba_predicted_label,confidence,proba_conflict,...,human_checked,corrected_aspect,corrected_polarity,keep,comment,threshold,timestamp,aspect_sentiment,aspect_polarity,aspect_category
0,0,Lilette,0,"We ate dinner here on Oct. 29, 2014. This res...",3.0,dinner,negative,negative,0.627231,0.063512,...,False,dinner,negative,1,NaN,0.6,2026-07-28T12:45:05.104258,dinner_negative,dinner_negative,specific_dish
1,1,Lilette,0,"We ate dinner here on Oct. 29, 2014. This res...",3.0,restaurant,negative,negative,0.664401,0.062805,...,False,restaurant,negative,1,NaN,0.6,2026-07-28T12:45:05.104278,restaurant_negative,restaurant_negative,restaurant_general
2,2,Lilette,0,"We ate dinner here on Oct. 29, 2014. This res...",3.0,restaurant,negative,negative,0.664401,0.062805,...,False,restaurant,negative,1,NaN,0.6,2026-07-28T12:45:05.104285,restaurant_negative,restaurant_negative,restaurant_general
3,3,Lilette,0,"We ate dinner here on Oct. 29, 2014. This res...",3.0,food,negative,negative,0.642056,0.063764,...,False,food,negative,1,NaN,0.6,2026-07-28T12:45:05.104291,food_negative,food_negative,food_quality
4,4,Lilette,0,"We ate dinner here on Oct. 29, 2014. This res...",3.0,atmosphere,negative,negative,0.633703,0.076215,...,False,atmosphere,negative,1,NaN,0.6,2026-07-28T12:45:05.104296,atmosphere_negative,atmosphere_negative,atmosphere


In [5]:
df.columns

Index(['Unnamed: 0', 'restaurant', 'id', 'text', 'stars', 'aspect', 'polarity',
       'proba_predicted_label', 'confidence', 'proba_conflict',
       'proba_negative', 'proba_neutral', 'proba_positive', 'llm_called',
       'llm_raw_response', 'llm_aspect', 'llm_polarity', 'llm_change_needed',
       'llm_reason', 'final_aspect', 'final_polarity', 'human_checked',
       'corrected_aspect', 'corrected_polarity', 'keep', 'comment',
       'threshold', 'timestamp', 'aspect_sentiment', 'aspect_polarity',
       'aspect_category'],
      dtype='object')

In [13]:
df[df["corrected_polarity"] == "negative"]["corrected_aspect"].value_counts().sum()

np.int64(2389)

In [30]:
df["aspect_category"].unique()

array(['specific_dish', 'restaurant_general', 'food_quality',
       'atmosphere', 'service', 'drinks', 'menu_variety',
       'seating_comfort', 'price_value', 'staff_behavior', 'desserts',
       'location_access', 'waiting_time', 'reservation', 'cleanliness'],
      dtype=object)

In [ ]:
import os
import json
import pandas as pd
from openai import OpenAI

os.environ["OPENAI_API_KEY"] = "YOUR_API_KEY"

client = OpenAI()

CATEGORIES = [
    "specific_dish",
    "restaurant_general",
    "food_quality",
    "atmosphere",
    "service",
    "drinks",
    "menu_variety",
    "seating_comfort",
    "price_value",
    "staff_behavior",
    "desserts",
    "location_access",
    "waiting_time",
    "reservation",
    "cleanliness",
    "others"
]

unique_aspects = (
    df["corrected_aspect"]
    .dropna()
    .astype(str)
    .str.strip()
    .drop_duplicates()
    .tolist()
)

def classify_batch(aspects):
    prompt = f"""
Classify each restaurant-review aspect into exactly one category.

Categories:
{", ".join(CATEGORIES)}

Definitions:
- specific_dish: a named dish or food item
- restaurant_general: the restaurant or dining experience generally
- food_quality: taste, freshness, cooking, portions or food presentation
- atmosphere: ambience, decor, music, noise or lighting
- service: general customer service or order handling
- drinks: beverages, coffee, tea, cocktails, wine or beer
- menu_variety: menu choices, selection or dietary options
- seating_comfort: chairs, tables, space or seating comfort
- price_value: price, affordability or value for money
- staff_behavior: friendliness, politeness, rudeness or professionalism
- desserts: cakes, sweets, ice cream or desserts
- location_access: location, parking, transport or accessibility
- waiting_time: delays or waiting for food, service or a table
- reservation: bookings or table reservations
- cleanliness: hygiene, bathrooms, tables or utensils
- others: anything that does not fit another category

Return one category for every aspect.

Aspects:
{json.dumps(aspects, ensure_ascii=False)}
"""

    response = client.responses.create(
        model="gpt-5-mini",
        input=prompt,
        text={
            "format": {
                "type": "json_schema",
                "name": "aspect_categories",
                "strict": True,
                "schema": {
                    "type": "object",
                    "properties": {
                        "results": {
                            "type": "array",
                            "items": {
                                "type": "object",
                                "properties": {
                                    "aspect": {"type": "string"},
                                    "category": {
                                        "type": "string",
                                        "enum": CATEGORIES
                                    }
                                },
                                "required": ["aspect", "category"],
                                "additionalProperties": False
                            }
                        }
                    },
                    "required": ["results"],
                    "additionalProperties": False
                }
            }
        }
    )

    return json.loads(response.output_text)["results"]

In [ ]:
all_results = []

batch_size = 50

for i in range(0, len(unique_aspects), batch_size):
    batch = unique_aspects[i:i + batch_size]
    all_results.extend(classify_batch(batch))

In [ ]:
aspect_mapping = {
    item["aspect"]: item["category"]
    for item in all_results
}

df["aspect_category"] = (
    df["corrected_aspect"]
    .astype("string")
    .str.strip()
    .map(aspect_mapping)
    .fillna("others")
)

In [ ]:
df[["corrected_aspect", "aspect_category"]].head()

In [ ]:
df.to_csv("categorized_reviews.csv", index=False)